In [ ]:
from collections import defaultdict, Counter
import math

In [ ]:
import json 

with open("/data_retrieval/kochwiki_data/rezepte_parsed.json", "r") as f:
    recipes = json.load(f)

with open("/data_retrieval/kochwiki_data/zutaten_parsed.json", "r") as f:
    ingredients = json.load(f)

data = recipes + ingredients

In [24]:
def simple_tokenize(text: str):
  return text.lower().split()

In [25]:
inverted_index = defaultdict(list)

corpus = {}
for i, r in enumerate(recipes):
    doc_id = f"recipe_{i}"
    corpus[doc_id] = {
        "title": r["title"],
        "text": r["plaintext"],
        "type": "recipe"
    }

for i, z in enumerate(ingredients):
    doc_id = f"ingredient_{i}"
    corpus[doc_id] = {
        "title": z["title"],
        "text": z["plaintext"],
        "type": "ingredient"
    }

for doc_id, doc in corpus.items():
    tokens = simple_tokenize(doc["text"])
    token_counts = Counter(tokens)
    for token, count in token_counts.items():
        inverted_index[token].append((doc_id, count))

print(f"Corpus size: {len(corpus)} documents")
print(f"Inverted index size: {len(inverted_index)} tokens")

Corpus size: 18114 documents
Inverted index size: 131645 tokens


In [26]:
from rank_bm25 import BM25Okapi

tokenized_corpus = [simple_tokenize(doc["text"]) for doc in corpus.values()]
bm25 = BM25Okapi(tokenized_corpus)

def bm25_search(query, n=5):
    tokenized_query = simple_tokenize(query)
    top_n = bm25.get_top_n(tokenized_query, list(corpus.keys()), n)
    return top_n

results = bm25_search("Tomaten und Basilikum")
for i, res in enumerate(results):
    print(f"Rank {i+1}: {corpus[res].get('title')}")

Rank 1: Brot mit eingelegten Tomaten
Rank 2: Tomatentatar
Rank 3: Crostini mit Ricotta und Tomaten
Rank 4: Paprika in Tomatensauce
Rank 5: Spaghetti alla trapanese


In [27]:
evaluation = {
    "Kartoffelsuppe": ["Vegetarische Kartoffelsuppe", "Vegane Kartoffelsuppe", "Leipziger Kartoffelsuppe", "Leas Kartoffelsuppe", "Lauch-Kartoffelsuppe"],
    "Gulasch": ["Würstchengulasch mit Paprika", "Wurstgulasch", "Winterlicher Krautgulasch", "Wildschweingulasch mit Pilzen und Senfsauce", "Wildschweingulasch mit Pilzen"],
    "Schokoladentorte": ["Himmlische Schokoladentorte", "Schokoladentorte, himmlisch", "Schokoladentorte mit Mascarpone-Minze-Creme", "Maroni-Schokoladentorte", "Brigittes Schokoladentorte"],
    "Weihnachtsplätzchen": ["Walisische Weihnachtsplätzchen", "Waliser Weihnachtsplätzchen", "Detmolder Weihnachtsplätzchen", "Zimtsterne mit Mandeln", "Plätzchen zum Ausstechen"],
    "Brot mit Sauerteig": ["Sauerteigbrot ohne Hefe aus dem Backautomaten", "Sauerteigbrot (hefefrei) aus dem Backautomaten", "Römisches Sauerteigbrot", "Roggenbrot auf Sauerteigbasis", "Kartoffelbrot mit Sauerteig"],
    "japanische Suppe": ["Misosuppe mit Tofu", "Miso-Suppe mit Tofu", "Ramen (Japanische Nudelsuppe)", "Japanische Nudelsuppe", "Miso Shiru"],
    "Curry": ["Tomaten-Linsen-Curry", "Thailändisches Gemüsecurry", "Veganes Blumenkohlcurry", "Thai-Lychee-Curry", "Zanderfilet mit Apfel-Gemüse-Curry"],
    "schnelles Mittagessen mit Nudeln": ["Zitronennudeln", "Studenten-Nudelauflauf", "Tagliatelle in Schinken-Sahnesauce", "Tagliatelle mit Lachs-Sherrysauce", "Zha-Jiang Nudeln nach Peking-Art"],
    "französische Soße": ["Sauce ravigote", "Vinaigrette mit roter Paprikaschote", "Vinaigrette mit Feta", "Erdbeervinaigrette", "Walnusssoße"],
    "Rezepte mit Kartoffeln, Zwiebeln und Speck": ["Bratkartoffeln", "Bauernfrühstück", "Himmel und Erde", "Kartoffelpfanne mit Käse", "Döppekoche"],
}

print("EVALUATION:\n")
total_hits = 0
total_possible = 0

for query, expected in evaluation.items():
    results = bm25_search(query, n=5)
    retrieved_titles = [corpus[res].get("title") for res in results]
    
    hits = sum(1 for doc in expected if doc in retrieved_titles)
    total_hits += hits
    total_possible += len(expected)
    
    print(f"Query: '{query}'")
    print(f"  Hits: {hits}/5")
    print(f"  Retrieved: {retrieved_titles}")
    print(f"  Expected:  {expected}")
    print()

print(f"{'='*50}")
print(f"Total: {total_hits}/{total_possible} relevant documents found")
print(f"Recall@5: {total_hits/total_possible:.1%}")

EVALUATION:

Query: 'Kartoffelsuppe'
  Hits: 2/5
  Retrieved: ['Kartoffelsuppe (vegan)', 'Vegetarische Kartoffelsuppe', 'Vegane Kartoffelsuppe', 'Kartoffelsuppe fein', 'Kartoffelsuppe mit Rauchwurst']
  Expected:  ['Vegetarische Kartoffelsuppe', 'Vegane Kartoffelsuppe', 'Leipziger Kartoffelsuppe', 'Leas Kartoffelsuppe', 'Lauch-Kartoffelsuppe']

Query: 'Gulasch'
  Hits: 0/5
  Retrieved: ['Provenzalisches Gulasch', 'Pörkölt', 'Exotisches Gulasch nach Großmutter-Art (Rimpf)', 'Zutat:Kalbsgulaschfleisch', 'Gulasch mit Bauernsalat']
  Expected:  ['Würstchengulasch mit Paprika', 'Wurstgulasch', 'Winterlicher Krautgulasch', 'Wildschweingulasch mit Pilzen und Senfsauce', 'Wildschweingulasch mit Pilzen']

Query: 'Schokoladentorte'
  Hits: 2/5
  Retrieved: ['Französischer Schokoladekuchen', 'Schokoladentorte mit Mascarpone-Minze-Creme', 'Brigittes Schokoladentorte', 'Schokoladentarte', 'Weiße Schokoladentorte']
  Expected:  ['Himmlische Schokoladentorte', 'Schokoladentorte, himmlisch', 'Schokola

In [28]:
queries_with_hit = 0
for query, expected in evaluation.items():
    results = bm25_search(query, n=5)
    retrieved_titles = [corpus[res].get("title") for res in results]
    if any(doc in retrieved_titles for doc in expected):
        queries_with_hit += 1
        
print(f"{queries_with_hit}/10 queries return at least one relevant document")

6/10 queries return at least one relevant document
